# CPI_通货膨胀

使用 `output_宏观_pkl/M0000612.pkl` 作为 CPI 当月同比原始数据，分别构造原始值、HP 平滑、Hamilton 回归、BK 带通滤波、CF 带通滤波版本，并用两状态高/低均值区制切换模型估计通货膨胀状态。

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = Path.cwd()
PKL_DIR = ROOT / "output_宏观_pkl"
OUT = ROOT / "cpi_inflation_regime_switching_output"
OUT.mkdir(exist_ok=True)

# output_宏观_pkl 中的 pkl 若不带日期索引，沿用现有 notebook 的口径：
# 从 2026-05-31 开始向前排列为月度数据。
START_DATE = pd.Timestamp("2026-05-31")
CPI_CODE = "M0000612"
CPI_NAME = "中国:CPI:当月同比"

# 月度景气周期常用 1.5 到 8 年窗口，即 18 到 96 个月。
BANDPASS_LOW = 18
BANDPASS_HIGH = 96
BK_K = 12

# 敏感度参数：状态持续概率越高，模型越强调历史惯性；sigma 越大，单期数据冲击越弱。
# 如果结果过于平滑、不容易提炼周期，优先提高 MIN_SWITCH_PROB 或降低 MAX_STAY_PROB，
# 再适度降低 LIKELIHOOD_SIGMA_SCALE。
TRANSITION_STAY_INIT = 0.80
MIN_SWITCH_PROB = 0.08
MAX_STAY_PROB = 1.0 - MIN_SWITCH_PROB
LIKELIHOOD_SIGMA_SCALE = 0.85
STATE_PROB_THRESHOLD = 0.50


## 读取原始 CPI 数据

In [3]:
def _first_numeric_series(obj) -> pd.Series:
    if isinstance(obj, pd.Series):
        return pd.to_numeric(obj, errors="coerce")

    if isinstance(obj, pd.DataFrame):
        numeric = obj.apply(pd.to_numeric, errors="coerce")
        cols = [col for col in numeric.columns if numeric[col].notna().any()]
        if not cols:
            raise ValueError("DataFrame 中没有可转成数值的列")
        return numeric[cols[0]]

    return pd.to_numeric(pd.Series(obj), errors="coerce")


def read_monthly_pkl(code: str, start_date: pd.Timestamp = START_DATE) -> pd.Series:
    path = PKL_DIR / f"{code}.pkl"
    if not path.exists():
        raise FileNotFoundError(path)

    raw = _first_numeric_series(pd.read_pickle(path)).dropna()
    idx_text = pd.Series(raw.index.astype(str)).str.strip()
    parsed_idx = pd.to_datetime(idx_text, errors="coerce")

    if parsed_idx.notna().any():
        s = pd.Series(raw.to_numpy(), index=parsed_idx)
        s = s[s.index.notna()]
    else:
        dates = [start_date - pd.offsets.MonthEnd(i) for i in range(len(raw))]
        s = pd.Series(raw.to_numpy(), index=pd.DatetimeIndex(dates))

    month_end = pd.DatetimeIndex(s.index).to_period("M").to_timestamp("M")
    monthly = pd.Series(s.to_numpy(), index=month_end).sort_index().groupby(level=0).last()
    regular = monthly.reindex(pd.date_range(monthly.index.min(), monthly.index.max(), freq="ME"))
    return regular.interpolate(limit=2).dropna().rename(code)


raw_cpi = read_monthly_pkl(CPI_CODE)

overview = pd.DataFrame([{
    "code": CPI_CODE,
    "name": CPI_NAME,
    "start": raw_cpi.index.min(),
    "end": raw_cpi.index.max(),
    "nobs": len(raw_cpi),
    "last_value": raw_cpi.iloc[-1],
}])

display(overview)
display(raw_cpi.to_frame("raw_cpi").tail(12))


C:\Users\16492\AppData\Local\Temp\ipykernel_20980\2620738510.py:22: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_idx = pd.to_datetime(idx_text, errors="coerce")


,code,name,start,end,nobs,last_value
0,M0000612,中国:CPI:当月同比,1987-01-31,2026-05-31,473,1.2


,raw_cpi
2025-06-30,0.1
2025-07-31,0.0
2025-08-31,-0.4
2025-09-30,-0.3
2025-10-31,0.2
2025-11-30,0.7
2025-12-31,0.8
2026-01-31,0.2
2026-02-28,1.3
2026-03-31,1.0


## HP / Hamilton / BK / CF 处理

In [4]:
def hp_filter(y: pd.Series, lamb: float = 129600.0) -> tuple[pd.Series, pd.Series]:
    """HP filter for monthly data. Returns cycle and smooth trend."""
    values = y.astype(float).to_numpy()
    n = len(values)
    if n < 4:
        raise ValueError("HP filter 至少需要 4 个观测值")

    eye = np.eye(n)
    d = np.zeros((n - 2, n))
    for i in range(n - 2):
        d[i, i] = 1.0
        d[i, i + 1] = -2.0
        d[i, i + 2] = 1.0

    trend_values = np.linalg.solve(eye + lamb * d.T @ d, values)
    trend = pd.Series(trend_values, index=y.index, name="hp_trend")
    cycle = (y - trend).rename("hp_cycle")
    return cycle, trend


def hamilton_filter(y: pd.Series, h: int = 8, p: int = 4) -> tuple[pd.Series, pd.Series]:
    """Hamilton regression detrending. Returns cycle and fitted trend."""
    x = y.astype(float).dropna()
    rows = []
    targets = []
    dates = []
    for t in range(p - 1, len(x) - h):
        rows.append([1.0] + [x.iloc[t - lag] for lag in range(p)])
        targets.append(x.iloc[t + h])
        dates.append(x.index[t + h])

    if len(rows) <= p:
        raise ValueError("Hamilton 回归样本不足")

    xmat = np.asarray(rows, dtype=float)
    yvec = np.asarray(targets, dtype=float)
    beta = np.linalg.lstsq(xmat, yvec, rcond=None)[0]
    fitted = pd.Series(xmat @ beta, index=pd.DatetimeIndex(dates), name="hamilton_trend")
    cycle = (x.reindex(fitted.index) - fitted).rename("hamilton_cycle")
    return cycle, fitted


def bk_filter(y: pd.Series, low: int = BANDPASS_LOW, high: int = BANDPASS_HIGH, k: int = BK_K) -> pd.Series:
    x = y.astype(float).dropna()
    if len(x) <= 2 * k + 1:
        raise ValueError("BK 滤波样本不足")

    omega_1 = 2.0 * np.pi / high
    omega_2 = 2.0 * np.pi / low
    weights = []
    for j in range(-k, k + 1):
        if j == 0:
            weight = (omega_2 - omega_1) / np.pi
        else:
            weight = (np.sin(j * omega_2) - np.sin(j * omega_1)) / (np.pi * j)
        weights.append(weight)

    weights = np.asarray(weights, dtype=float)
    weights = weights - weights.mean()
    filtered = np.convolve(x.to_numpy(), weights, mode="valid")
    index = x.index[k : len(x) - k]
    return pd.Series(filtered, index=index, name="bk_cycle")


def cf_filter(y: pd.Series, low: int = BANDPASS_LOW, high: int = BANDPASS_HIGH) -> pd.Series:
    """Christiano-Fitzgerald-style asymmetric band-pass approximation."""
    x = y.astype(float).dropna()
    values = x.to_numpy()
    n = len(values)
    if n < 8:
        raise ValueError("CF 滤波样本不足")

    demeaned = values - values.mean()
    omega_1 = 2.0 * np.pi / high
    omega_2 = 2.0 * np.pi / low
    cycle = np.zeros(n)

    for t in range(n):
        weights = np.zeros(n)
        for s in range(n):
            j = s - t
            if j == 0:
                weights[s] = (omega_2 - omega_1) / np.pi
            else:
                weights[s] = (np.sin(j * omega_2) - np.sin(j * omega_1)) / (np.pi * j)
        weights -= weights.mean()
        cycle[t] = weights @ demeaned

    return pd.Series(cycle, index=x.index, name="cf_cycle")


def build_model_inputs(raw: pd.Series) -> dict[str, pd.Series]:
    hp_cycle, hp_trend = hp_filter(raw)
    hamilton_cycle, hamilton_trend = hamilton_filter(raw)
    bk_cycle = bk_filter(raw)
    cf_cycle = cf_filter(raw)

    return {
        "raw": raw.rename("raw"),
        "hp_smooth_trend": hp_trend,
        "hp_cycle": hp_cycle,
        "hamilton_cycle": hamilton_cycle,
        "hamilton_trend": hamilton_trend,
        "bk_cycle": bk_cycle,
        "cf_cycle": cf_cycle,
    }


model_inputs = build_model_inputs(raw_cpi)
series_table = pd.concat(model_inputs, axis=1).rename_axis("date")
series_table.insert(0, "raw_cpi", raw_cpi.reindex(series_table.index))

display(series_table.tail(12))


,raw_cpi,raw,hp_smooth_trend,hp_cycle,hamilton_cycle,hamilton_trend,bk_cycle,cf_cycle
date,,,,,,,,
2025-06-30,0.1,0.1,0.257003,-0.157003,-0.971333,1.071333,NaN,-1.048540
2025-07-31,0.0,0.0,0.237399,-0.237399,-0.937566,0.937566,NaN,-1.146861
2025-08-31,-0.4,-0.4,0.218116,-0.618116,-1.319759,0.919759,NaN,-1.192650
2025-09-30,-0.3,-0.3,0.199119,-0.499119,-2.278923,1.978923,NaN,-1.177196
2025-10-31,0.2,0.2,0.180369,0.019631,1.044335,-0.844335,NaN,-1.095438
2025-11-30,0.7,0.7,0.161820,0.538180,-0.307541,1.007541,NaN,-0.946476
2025-12-31,0.8,0.8,0.143430,0.656570,-0.242562,1.042562,NaN,-0.733784
2026-01-31,0.2,0.2,0.125159,0.074841,-1.211890,1.411890,NaN,-0.465092
2026-02-28,1.3,1.3,0.106973,1.193027,-0.238012,1.538012,NaN,-0.151940


## 两状态区制切换模型

In [5]:
def _normal_pdf(y: np.ndarray, mu: np.ndarray, sigma: float) -> np.ndarray:
    sigma = max(float(sigma), 1e-8)
    z = (y[:, None] - mu[None, :]) / sigma
    return np.exp(-0.5 * z * z) / (np.sqrt(2.0 * np.pi) * sigma)


def _constrain_transition(transition: np.ndarray, min_switch_prob: float) -> np.ndarray:
    """限制状态持续概率，避免模型过度依赖历史惯性而迟迟不切换。"""
    constrained = np.asarray(transition, dtype=float).copy()
    min_switch_prob = float(np.clip(min_switch_prob, 0.0, 0.49))
    max_stay_prob = 1.0 - min_switch_prob

    for i in range(constrained.shape[0]):
        constrained[i, i] = min(constrained[i, i], max_stay_prob)
        off_diag = [j for j in range(constrained.shape[1]) if j != i]
        remaining = 1.0 - constrained[i, i]
        off_sum = constrained[i, off_diag].sum()
        if off_sum <= 0:
            constrained[i, off_diag] = remaining / len(off_diag)
        else:
            constrained[i, off_diag] = constrained[i, off_diag] / off_sum * remaining

    return constrained / constrained.sum(axis=1, keepdims=True)


def _forward_filter(
    values: np.ndarray,
    mu: np.ndarray,
    sigma: float,
    transition: np.ndarray,
    init_prob: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, float]:
    emission = _normal_pdf(values, mu, sigma)
    prior = np.zeros((len(values), 2))
    filtered = np.zeros((len(values), 2))
    scale = np.zeros(len(values))

    prior[0] = init_prob
    filtered[0] = prior[0] * emission[0]
    scale[0] = filtered[0].sum()
    filtered[0] /= scale[0]

    for t in range(1, len(values)):
        prior[t] = filtered[t - 1] @ transition
        filtered[t] = prior[t] * emission[t]
        scale[t] = filtered[t].sum()
        filtered[t] /= scale[t]

    return prior, filtered, float(np.log(scale).sum())


def fit_two_state_gaussian_hmm(
    y: pd.Series,
    max_iter: int = 500,
    tol: float = 1e-8,
    transition_stay_init: float = TRANSITION_STAY_INIT,
    min_switch_prob: float = MIN_SWITCH_PROB,
    likelihood_sigma_scale: float = LIKELIHOOD_SIGMA_SCALE,
    state_prob_threshold: float = STATE_PROB_THRESHOLD,
) -> tuple[pd.DataFrame, dict]:
    """Estimate a two-state Gaussian regime switching model with EM."""
    x = y.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    values = x.to_numpy()
    if len(values) < 12:
        raise ValueError("区制切换模型至少需要 12 个有效观测值")

    q25, q75 = np.nanpercentile(values, [25, 75])
    mu = np.asarray([q25, q75], dtype=float)
    sigma = float(np.nanstd(values, ddof=1)) or 1.0
    transition_stay_init = float(np.clip(transition_stay_init, 0.51, 0.99))
    transition = np.asarray(
        [[transition_stay_init, 1.0 - transition_stay_init], [1.0 - transition_stay_init, transition_stay_init]],
        dtype=float,
    )
    transition = _constrain_transition(transition, min_switch_prob)
    init_prob = np.asarray([0.50, 0.50], dtype=float)
    last_ll = -np.inf

    for iteration in range(1, max_iter + 1):
        emission = _normal_pdf(values, mu, sigma)
        alpha = np.zeros((len(values), 2))
        scale = np.zeros(len(values))

        alpha[0] = init_prob * emission[0]
        scale[0] = alpha[0].sum()
        alpha[0] /= scale[0]

        for t in range(1, len(values)):
            alpha[t] = (alpha[t - 1] @ transition) * emission[t]
            scale[t] = alpha[t].sum()
            alpha[t] /= scale[t]

        beta = np.ones((len(values), 2))
        for t in range(len(values) - 2, -1, -1):
            beta[t] = transition @ (emission[t + 1] * beta[t + 1])
            beta[t] /= scale[t + 1]

        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True)

        xi_sum = np.zeros((2, 2))
        for t in range(len(values) - 1):
            xi = (
                alpha[t, :, None]
                * transition
                * emission[t + 1, None, :]
                * beta[t + 1, None, :]
            )
            xi_sum += xi / xi.sum()

        init_prob = gamma[0]
        transition = xi_sum / xi_sum.sum(axis=1, keepdims=True)
        transition = _constrain_transition(transition, min_switch_prob)
        weights = gamma.sum(axis=0)
        mu = (gamma * values[:, None]).sum(axis=0) / weights
        sigma = np.sqrt(((gamma * (values[:, None] - mu[None, :]) ** 2).sum()) / len(values))

        ll = float(np.log(scale).sum())
        if abs(ll - last_ll) < tol:
            break
        last_ll = ll

    low_state = int(np.argmin(mu))
    high_state = int(np.argmax(mu))
    p_stay_high = float(transition[high_state, high_state])
    p_stay_low = float(transition[low_state, low_state])
    sigma_for_filter = max(float(sigma) * float(likelihood_sigma_scale), 1e-8)
    prior, filtered, filtered_ll = _forward_filter(values, mu, sigma_for_filter, transition, init_prob)

    out = pd.DataFrame(index=x.index)
    out["model_value"] = values
    out["prior_prob_low_state"] = prior[:, low_state]
    out["prior_prob_high_state"] = prior[:, high_state]
    out["prob_low_state"] = filtered[:, low_state]
    out["prob_high_state"] = filtered[:, high_state]
    out["state"] = np.where(out["prob_high_state"] >= state_prob_threshold, "高通胀状态", "低通胀状态")
    out["inflation_regime"] = np.where(out["prob_high_state"] >= state_prob_threshold, "通胀偏高", "通胀偏低")

    params = {
        "mu_low": float(mu[low_state]),
        "mu_high": float(mu[high_state]),
        "sigma": float(sigma),
        "sigma_for_filter": float(sigma_for_filter),
        "likelihood_sigma_scale": float(likelihood_sigma_scale),
        "p_stay_high": p_stay_high,
        "p_stay_low": p_stay_low,
        "p_switch_high_to_low": 1.0 - p_stay_high,
        "p_switch_low_to_high": 1.0 - p_stay_low,
        "min_switch_prob": float(min_switch_prob),
        "state_prob_threshold": float(state_prob_threshold),
        "log_likelihood": filtered_ll,
        "iterations": iteration,
        "nobs": len(values),
        "start": x.index.min(),
        "end": x.index.max(),
    }
    return out, params


## 运行模型并导出结果

In [6]:
series_table.to_csv(OUT / "cpi_inflation_model_inputs.csv", encoding="utf-8-sig")

state_tables = []
param_rows = []
latest_rows = []

for method, series in model_inputs.items():
    states, params = fit_two_state_gaussian_hmm(series)
    states = states.rename_axis("date").reset_index()
    states.insert(0, "method", method)
    states.insert(1, "code", CPI_CODE)
    states.insert(2, "name", CPI_NAME)
    state_tables.append(states)

    param_rows.append({"method": method, "code": CPI_CODE, "name": CPI_NAME, **params})
    latest = states.iloc[-1].to_dict()
    latest_rows.append({
        "method": method,
        "latest_state_date": latest["date"],
        "model_value": latest["model_value"],
        "prob_high_state": latest["prob_high_state"],
        "prob_low_state": latest["prob_low_state"],
        "state": latest["state"],
        "inflation_regime": latest["inflation_regime"],
        "mu_low": params["mu_low"],
        "mu_high": params["mu_high"],
        "p_stay_high": params["p_stay_high"],
        "p_stay_low": params["p_stay_low"],
    })

all_states = pd.concat(state_tables, ignore_index=True)
params_table = pd.DataFrame(param_rows)
latest_table = pd.DataFrame(latest_rows)

diagnostics_table = (
    all_states.sort_values(["method", "date"])
    .groupby("method")
    .apply(lambda x: pd.Series({
        "start": x["date"].min(),
        "end": x["date"].max(),
        "months": len(x),
        "state_switches": int(x["state"].ne(x["state"].shift()).sum() - 1),
        "avg_abs_prob_high_change": x["prob_high_state"].diff().abs().mean(),
        "low_confidence_months_40_60": int(x["prob_high_state"].between(0.40, 0.60).sum()),
    }))
    .reset_index()
)

sensitivity_guidance = pd.DataFrame([
    {
        "parameter": "MIN_SWITCH_PROB / MAX_STAY_PROB",
        "current": f"{MIN_SWITCH_PROB:.2f} / {MAX_STAY_PROB:.2f}",
        "role": "控制状态持续概率，也就是历史惯性；这是影响模型是否愿意切换状态的核心参数。",
        "more_sensitive": "提高 MIN_SWITCH_PROB 到 0.10-0.20，对应 MAX_STAY_PROB 降到 0.80-0.90。",
        "less_sensitive": "降低 MIN_SWITCH_PROB 到 0.03-0.08，对应 MAX_STAY_PROB 提高到 0.92-0.97。",
        "suggested_range": "0.05-0.20；CPI 月度数据若想更快识别通胀周期，可先试 0.10、0.15。",
    },
    {
        "parameter": "LIKELIHOOD_SIGMA_SCALE",
        "current": LIKELIHOOD_SIGMA_SCALE,
        "role": "缩放观测扰动 sigma；越小，当前数据和高/低均值的距离越容易改变后验概率。",
        "more_sensitive": "下调到 0.60-0.85。",
        "less_sensitive": "上调到 0.90-1.20。",
        "suggested_range": "0.60-1.00；过低会把噪声也当成状态切换。",
    },
    {
        "parameter": "HP lambda",
        "current": 129600,
        "role": "HP 趋势平滑强度；lambda 越低，趋势越贴近原始数据，周期项更敏感。",
        "more_sensitive": "改为 14400-60000。",
        "less_sensitive": "维持 129600 或更高。",
        "suggested_range": "月度数据常见 129600；若 CPI 转折识别过慢，可试 14400、30000、60000。",
    },
    {
        "parameter": "BANDPASS_LOW / BANDPASS_HIGH / BK_K",
        "current": f"{BANDPASS_LOW} / {BANDPASS_HIGH} / {BK_K}",
        "role": "带通窗口决定保留多长的周期；当前 18-96 个月偏中周期，短波动会被过滤。",
        "more_sensitive": "LOW 调到 6-12，HIGH 调到 36-60，BK_K 调到 6-9。",
        "less_sensitive": "维持 18-96，BK_K 维持 12 或更高。",
        "suggested_range": "短周期识别可试 9-48；中周期识别用 18-96。",
    },
    {
        "parameter": "Hamilton h / p",
        "current": "h=8, p=4",
        "role": "h 越小，趋势估计越短视，周期项越敏感；p 控制滞后信息长度。",
        "more_sensitive": "h 改为 4-6，p 改为 2-4。",
        "less_sensitive": "h 维持 8-12，p 维持 4。",
        "suggested_range": "CPI 可试 h=4,p=3 或 h=6,p=4。",
    },
])

all_states.to_csv(OUT / "cpi_inflation_regime_states_by_month.csv", index=False, encoding="utf-8-sig")
params_table.to_csv(OUT / "cpi_inflation_regime_model_params.csv", index=False, encoding="utf-8-sig")
latest_table.to_csv(OUT / "cpi_inflation_regime_latest_state.csv", index=False, encoding="utf-8-sig")
diagnostics_table.to_csv(OUT / "cpi_inflation_regime_diagnostics.csv", index=False, encoding="utf-8-sig")
sensitivity_guidance.to_csv(OUT / "cpi_inflation_sensitivity_guidance.csv", index=False, encoding="utf-8-sig")

excel_path = OUT / "cpi_inflation_regime_switching.xlsx"
method_sheet_names = {
    "raw": "raw",
    "hp_smooth_trend": "hp_smooth",
    "hp_cycle": "hp_cycle",
    "hamilton_cycle": "ham_cycle",
    "hamilton_trend": "ham_trend",
    "bk_cycle": "bk_cycle",
    "cf_cycle": "cf_cycle",
}

with pd.ExcelWriter(excel_path) as writer:
    series_table.reset_index().to_excel(writer, sheet_name="模型输入序列", index=False)
    all_states.to_excel(writer, sheet_name="逐月状态概率", index=False)
    params_table.to_excel(writer, sheet_name="模型参数", index=False)
    latest_table.to_excel(writer, sheet_name="最新状态", index=False)
    diagnostics_table.to_excel(writer, sheet_name="模型诊断", index=False)
    sensitivity_guidance.to_excel(writer, sheet_name="参数修改建议", index=False)
    for method, sheet_name in method_sheet_names.items():
        method_states = all_states[all_states["method"].eq(method)].copy()
        method_states.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"已导出到：{OUT}")
print(f"Excel 文件：{excel_path}")
display(latest_table)


C:\Users\16492\AppData\Local\Temp\ipykernel_20980\4090908570.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


已导出到：c:\Users\16492\Desktop\实习内容\数据处理\data\cpi_inflation_regime_switching_output
Excel 文件：c:\Users\16492\Desktop\实习内容\数据处理\data\cpi_inflation_regime_switching_output\cpi_inflation_regime_switching.xlsx


,method,latest_state_date,model_value,prob_high_state,prob_low_state,state,inflation_regime,mu_low,mu_high,p_stay_high,p_stay_low
0,raw,2026-05-31,1.200000,1.652965e-13,1.000000,低通胀状态,通胀偏低,2.502928,20.484624,0.920000,0.92
1,hp_smooth_trend,2026-05-31,0.052630,2.380392e-24,1.000000,低通胀状态,通胀偏低,1.937973,11.397550,0.920000,0.92
2,hp_cycle,2026-05-31,1.147370,4.060396e-06,0.999996,低通胀状态,通胀偏低,-0.812346,11.222220,0.920000,0.92
3,hamilton_cycle,2026-05-31,0.565945,2.498333e-04,0.999750,低通胀状态,通胀偏低,-0.755203,7.930125,0.919583,0.92
4,hamilton_trend,2026-05-31,0.634055,5.356767e-12,1.000000,低通胀状态,通胀偏低,2.830880,17.706076,0.920000,0.92
5,bk_cycle,2025-05-31,-0.424724,1.395780e-07,1.000000,低通胀状态,通胀偏低,-0.271719,4.578933,0.899576,0.92
6,cf_cycle,2026-05-31,0.899017,4.426169e-06,0.999996,低通胀状态,通胀偏低,-0.752044,11.184171,0.920000,0.92


In [8]:
display(params_table)
display(diagnostics_table)
display(sensitivity_guidance)
display(all_states.tail(20))


,method,code,name,mu_low,mu_high,sigma,sigma_for_filter,likelihood_sigma_scale,p_stay_high,p_stay_low,p_switch_high_to_low,p_switch_low_to_high,min_switch_prob,state_prob_threshold,log_likelihood,iterations,nobs,start,end
0,raw,M0000612,中国:CPI:当月同比,2.502928,20.484624,3.081008,2.618857,0.85,0.920000,0.92,0.080000,0.08,0.08,0.5,-1262.474657,17,473,1987-01-31,2026-05-31
1,hp_smooth_trend,M0000612,中国:CPI:当月同比,1.937973,11.397550,1.291169,1.097494,0.85,0.920000,0.92,0.080000,0.08,0.08,0.5,-846.590350,8,473,1987-01-31,2026-05-31
2,hp_cycle,M0000612,中国:CPI:当月同比,-0.812346,11.222220,2.603393,2.212884,0.85,0.920000,0.92,0.080000,0.08,0.08,0.5,-1183.900945,18,473,1987-01-31,2026-05-31
3,hamilton_cycle,M0000612,中国:CPI:当月同比,-0.755203,7.930125,2.490081,2.116568,0.85,0.919583,0.92,0.080417,0.08,0.08,0.5,-1137.773327,19,462,1987-12-31,2026-05-31
4,hamilton_trend,M0000612,中国:CPI:当月同比,2.830880,17.706076,2.904665,2.468965,0.85,0.920000,0.92,0.080000,0.08,0.08,0.5,-1207.120809,15,462,1987-12-31,2026-05-31
5,bk_cycle,M0000612,中国:CPI:当月同比,-0.271719,4.578933,1.139030,0.968175,0.85,0.899576,0.92,0.100424,0.08,0.08,0.5,-754.831492,22,449,1988-01-31,2025-05-31
6,cf_cycle,M0000612,中国:CPI:当月同比,-0.752044,11.184171,2.685995,2.283096,0.85,0.920000,0.92,0.080000,0.08,0.08,0.5,-1197.522182,28,473,1987-01-31,2026-05-31


,method,start,end,months,state_switches,avg_abs_prob_high_change,low_confidence_months_40_60
0,bk_cycle,1988-01-31,2025-05-31,449,6,0.013412,2
1,cf_cycle,1987-01-31,2026-05-31,473,4,0.009748,3
2,hamilton_cycle,1987-12-31,2026-05-31,462,8,0.022781,2
3,hamilton_trend,1987-12-31,2026-05-31,462,4,0.009302,1
4,hp_cycle,1987-01-31,2026-05-31,473,4,0.009106,1
5,hp_smooth_trend,1987-01-31,2026-05-31,473,1,0.002119,0
6,raw,1987-01-31,2026-05-31,473,4,0.008478,1


,parameter,current,role,more_sensitive,less_sensitive,suggested_range
0,MIN_SWITCH_PROB / MAX_STAY_PROB,0.08 / 0.92,控制状态持续概率，也就是历史惯性；这是影响模型是否愿意切换状态的核心参数。,提高 MIN_SWITCH_PROB 到 0.10-0.20，对应 MAX_STAY_PRO...,降低 MIN_SWITCH_PROB 到 0.03-0.08，对应 MAX_STAY_PRO...,0.05-0.20；CPI 月度数据若想更快识别通胀周期，可先试 0.10、0.15。
1,LIKELIHOOD_SIGMA_SCALE,0.85,缩放观测扰动 sigma；越小，当前数据和高/低均值的距离越容易改变后验概率。,下调到 0.60-0.85。,上调到 0.90-1.20。,0.60-1.00；过低会把噪声也当成状态切换。
2,HP lambda,129600,HP 趋势平滑强度；lambda 越低，趋势越贴近原始数据，周期项更敏感。,改为 14400-60000。,维持 129600 或更高。,月度数据常见 129600；若 CPI 转折识别过慢，可试 14400、30000、60000。
3,BANDPASS_LOW / BANDPASS_HIGH / BK_K,18 / 96 / 12,带通窗口决定保留多长的周期；当前 18-96 个月偏中周期，短波动会被过滤。,LOW 调到 6-12，HIGH 调到 36-60，BK_K 调到 6-9。,维持 18-96，BK_K 维持 12 或更高。,短周期识别可试 9-48；中周期识别用 18-96。
4,Hamilton h / p,"h=8, p=4",h 越小，趋势估计越短视，周期项越敏感；p 控制滞后信息长度。,h 改为 4-6，p 改为 2-4。,h 维持 8-12，p 维持 4。,"CPI 可试 h=4,p=3 或 h=6,p=4。"


,method,code,name,date,model_value,prior_prob_low_state,prior_prob_high_state,prob_low_state,prob_high_state,state,inflation_regime
3245,cf_cycle,M0000612,中国:CPI:当月同比,2024-10-31,0.087169,0.919999,0.080001,0.999999,6.896749e-07,低通胀状态,通胀偏低
3246,cf_cycle,M0000612,中国:CPI:当月同比,2024-11-30,0.022017,0.919999,0.080001,0.999999,5.940885e-07,低通胀状态,通胀偏低
3247,cf_cycle,M0000612,中国:CPI:当月同比,2024-12-31,-0.082419,0.920000,0.080000,1.000000,4.677236e-07,低通胀状态,通胀偏低
3248,cf_cycle,M0000612,中国:CPI:当月同比,2025-01-31,-0.220802,0.920000,0.080000,1.000000,3.406965e-07,低通胀状态,通胀偏低
3249,cf_cycle,M0000612,中国:CPI:当月同比,2025-02-28,-0.384700,0.920000,0.080000,1.000000,2.340837e-07,低通胀状态,通胀偏低
3250,cf_cycle,M0000612,中国:CPI:当月同比,2025-03-31,-0.563013,0.920000,0.080000,1.000000,1.556105e-07,低通胀状态,通胀偏低
3251,cf_cycle,M0000612,中国:CPI:当月同比,2025-04-30,-0.742631,0.920000,0.080000,1.000000,1.031359e-07,低通胀状态,通胀偏低
3252,cf_cycle,M0000612,中国:CPI:当月同比,2025-05-31,-0.909284,0.920000,0.080000,1.000000,7.041636e-08,低通胀状态,通胀偏低
3253,cf_cycle,M0000612,中国:CPI:当月同比,2025-06-30,-1.048540,0.920000,0.080000,1.000000,5.118988e-08,低通胀状态,通胀偏低
3254,cf_cycle,M0000612,中国:CPI:当月同比,2025-07-31,-1.146861,0.920000,0.080000,1.000000,4.086998e-08,低通胀状态,通胀偏低
